In [ ]:
# -*- coding: utf-8 -*-

import os
import pandas as pd
import matplotlib.pyplot as plt

# 기본 경로 (필요시 바꾸세요)
BASE_DIR = os.path.expanduser('~/ros2_ws/src/test_truck')
FILES = [
    ('Truck 1', os.path.join(BASE_DIR, '1_truck.csv')),
    ('Truck 2', os.path.join(BASE_DIR, '2_truck.csv')),
    ('Truck 3', os.path.join(BASE_DIR, '3_truck.csv')),
]

def load_csv(path):
    """CSV 로드: timestamp,x,y,velocity,pitch 헤더 가정"""
    if not os.path.exists(path):
        raise FileNotFoundError(f"CSV not found: {path}")
    df = pd.read_csv(path)
    # 컬럼 존재 체크 및 타입 변환
    for col in ['timestamp', 'x', 'y', 'velocity']:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' missing in {path}")
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
    df['x'] = pd.to_numeric(df['x'], errors='coerce')
    df['y'] = pd.to_numeric(df['y'], errors='coerce')
    df['velocity'] = pd.to_numeric(df['velocity'], errors='coerce')
    df = df.dropna(subset=['timestamp', 'x', 'y', 'velocity']).reset_index(drop=True)
    # 시작 시점을 0초로 정규화(보기 편하게)
    if len(df) > 0:
        t0 = df['timestamp'].iloc[0]
        df['t_rel'] = df['timestamp'] - t0
    else:
        df['t_rel'] = df['timestamp']
    return df

def plot_xy(dfs):
    """x-y 궤적을 하나의 그림에 플롯 (트럭 3대)"""
    plt.figure()
    for name, df in dfs:
        if len(df) == 0:
            continue
        plt.plot(df['x'], df['y'], label=name)  # 색상은 지정하지 않음(기본)
    plt.xlabel('X (m)')
    plt.ylabel('Y (m)')
    plt.title('Truck Trajectories (X-Y)')
    plt.legend()
    plt.axis('equal')  # 스케일 동일
    plt.grid(True)
    plt.tight_layout()
    # 저장도 원하면 주석 해제
    # plt.savefig(os.path.join(BASE_DIR, 'trajectories_xy.png'), dpi=200)

def plot_speed_time(dfs):
    """속도-시간 그래프(상대시간 t_rel, 초)"""
    plt.figure()
    for name, df in dfs:
        if len(df) == 0:
            continue
        plt.plot(df['t_rel'], df['velocity'], label=name)  # 색상 지정 안함
    plt.xlabel('Time (s)')
    plt.ylabel('Speed (m/s)')
    plt.title('Truck Speed vs Time')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    # 저장도 원하면 주석 해제
    # plt.savefig(os.path.join(BASE_DIR, 'speed_time.png'), dpi=200)

def main():
    # CSV 로드
    dfs = []
    for name, path in FILES:
        try:
            df = load_csv(path)
            dfs.append((name, df))
            print(f"Loaded {name}: {len(df)} rows from {path}")
        except Exception as e:
            print(f"[WARN] {name} skipped: {e}")

    if not dfs:
        print("No data to plot. Check CSV paths.")
        return

    # 1) x-y 궤적
    plot_xy(dfs)
    # 2) 속도-시간
    plot_speed_time(dfs)

    # 화면 표시
    plt.show()

if __name__ == '__main__':
    main()
